# 3. Portfolio Construction and Risk Adjustment

## Overview
This notebook implements the portfolio construction process across five asset classes with dynamic, risk-adjusted weight allocation:

1. **Strategic Asset Class Allocation**
   - Equities: 65% | Bonds: 10% | Precious Metals: 5% | Energy: 5% | Agriculture: 5%
   - Regional diversification within equities and bonds

2. **Sharpe Ratio Adjustments**
   - Weights adjusted up or down based on risk-adjusted returns vs. each asset class benchmark
   - Per-class sensitivity controls how much the Sharpe ratio can shift weights:
     - Equities: ±0.1 | Bonds: ±0.25 | Precious Metals: ±0.15 | Energy: ±0.15 | Agriculture: ±0.15

3. **Final Portfolio Construction**
   - Risk-adjusted position sizing across all five asset classes
   - Volatility-based cash weight adjustments
   - Final investment allocation in GBP

4. **Portfolio Versioning**
   - 2025 portfolio (equity + bonds only) is seeded from `final_portfolio_25.csv` and **locked** on first run
   - 2026 portfolio (all five asset classes) is saved to the `portfolios` table and remains **mutable** until explicitly locked

The process aims to create a well-diversified portfolio that balances return potential with risk management.

## Setup and Data Loading

In [1]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path("..").resolve()))

import pandas as pd
import numpy as np

from etf_utils.config import DATA_RAW, DATA_INTERMEDIATE, DATA_OUTPUT, DATA_CONFIG
from etf_utils.data_provider import DataProvider
from etf_utils.data_io import load_intermediate, save_output
from etf_utils.metrics import calculate_annualized_volatility, interpolate_adjustment_factor
from etf_utils.database import load_screened_etfs, save_portfolio, seed_2025_portfolio

provider = DataProvider()

# Seed the locked 2025 portfolio (no-op if already done)
#seed_2025_portfolio()

# Load screened ETFs for 2026 from the DB (falls back to summary_all.csv if DB is empty)
etfs_df = load_screened_etfs(portfolio_year=2026)
if etfs_df.empty:
    etfs_df = pd.read_csv(DATA_INTERMEDIATE / 'summary_all.csv')

# Guard: drop any stale asset classes left over from a prior portfolio model run
# (e.g. 'energy' / 'agri' rows from the old 5-class model).  Only the four current
# classes are valid; anything else would cause a KeyError in sr_data_map below.
_EXPECTED_CLASSES = {'equity', 'bonds', 'preciousMetals', 'commodities'}
_stale = set(etfs_df['asset_class'].unique()) - _EXPECTED_CLASSES
if _stale:
    import warnings
    warnings.warn(
        f"Dropping stale asset classes from screened ETFs: {sorted(_stale)}. "
        "Re-run notebook 02 to refresh the DB, or manually delete via "
        "etf_utils.database.purge_screened_etfs_for_year(2026).",
        stacklevel=1,
    )
    etfs_df = etfs_df[etfs_df['asset_class'].isin(_EXPECTED_CLASSES)].copy()

print(f"Loaded {len(etfs_df)} ETFs for portfolio construction (asset classes: {etfs_df['asset_class'].unique().tolist()})")
etfs_df

Loaded 20 ETFs for portfolio construction (asset classes: ['equity', 'bonds', 'preciousMetals', 'commodities'])


,wkn,ticker,valor,name,inception_date,age_in_days,age_in_years,strategy,domicile_country,currency,...,available_on_investengine,yday_close_price_date,yday_close_price,metal_type,commodity_overlap_pct,within_metal_beta,tracking_fidelity,sector,benchmark_return,platform
0,A0HGV6,IUKD,2308843.0,iShares UK Dividend UCITS ETF,2005-11-04 00:00:00,7463.0,20.446575,Long-only,Ireland,GBP,...,None,2026-04-10,9.9060,NaN,None,None,None,None,None,InvestEngine
1,A1W9KD,XSTC,38111441.0,Xtrackers MSCI USA Information Technology UCIT...,2017-09-12 00:00:00,3133.0,8.583562,Long-only,Ireland,USD,...,None,2026-04-10,100.8600,NaN,None,None,None,None,None,InvestEngine
2,A0Q1YX,ISJP,4218299.0,iShares MSCI Japan Small Cap UCITS ETF (Dist),2008-05-09 00:00:00,6546.0,17.934247,Long-only,Ireland,USD,...,None,2026-04-10,42.8700,NaN,None,None,None,None,None,InvestEngine
3,A0MZWP,IMIB,3246482.0,iShares FTSE MIB UCITS ETF EUR (Dist),2007-07-06 00:00:00,6854.0,18.778082,Long-only,Ireland,EUR,...,None,2026-04-10,25.2225,NaN,None,None,None,None,None,InvestEngine
4,A2JF6S,VGER,41860972.0,Vanguard Germany All Cap UCITS ETF (EUR) Distr...,2018-07-17 00:00:00,2825.0,7.739726,Long-only,Ireland,EUR,...,None,2026-04-10,29.7125,NaN,None,None,None,None,None,InvestEngine
5,A0HGWA,IBZL,2308866.0,iShares MSCI Brazil UCITS ETF (Dist),2005-11-18 00:00:00,7449.0,20.408219,Long-only,Ireland,USD,...,None,2026-04-10,24.6500,NaN,None,None,None,None,None,InvestEngine
6,A1JHYT,HMCH,12534159.0,HSBC MSCI China UCITS ETF USD,2011-01-26 00:00:00,5554.0,15.216438,Long-only,Ireland,USD,...,None,2026-04-10,5.8450,NaN,None,None,None,None,None,InvestEngine
7,A1JJU5,HKOR,12843012.0,HSBC MSCI Korea Capped UCITS ETF USD,2011-04-06 00:00:00,5484.0,15.024658,Long-only,Ireland,USD,...,None,2026-04-10,86.4850,NaN,None,None,None,None,None,InvestEngine
8,A0LGP9,IGLT,2803931.0,iShares Core UK Gilts UCITS ETF,2006-12-01 00:00:00,7071.0,19.372603,Long-only,Ireland,GBP,...,None,2026-04-10,9.8275,NaN,None,None,None,None,None,InvestEngine
9,A0DKL3,SLXX,1828476.0,iShares Core GBP Corporate Bond UCITS ETF,2004-03-29 00:00:00,8048.0,22.049315,Long-only,Ireland,GBP,...,None,2026-04-09,120.9300,NaN,None,None,None,None,None,InvestEngine


## Risk Adjustment Framework

### Sharpe Ratio Adjustment System
We implement a dynamic weight adjustment system based on Sharpe ratios relative to each asset class benchmark:

1. **Base Factors**: Scale from 0.6 (poor) to 1.48 (excellent)
2. **Relative Performance**: Adjustments based on deviation from the intra-class median Sharpe ratio
3. **Asset Class Sensitivity** (maximum shift in adjustment factor):
   - Equities: ±0.1
   - Bonds: ±0.25
   - Precious Metals: ±0.15
   - Energy: ±0.15
   - Agriculture: ±0.15

This allows the portfolio to:
- Reward strong risk-adjusted performance
- Reduce exposure to high-risk/low-return assets
- Apply appropriate sensitivity by asset class

In [2]:
# Sharpe ratio adjustment factors (top-level, across all asset classes)
trailing_sr_adjustment_factors = {
    -1.0: 0.60,
    -0.8: 0.66,
    -0.6: 0.77,
    -0.4: 0.85,
    -0.2: 0.94,
     0.0: 1.00,
     0.2: 1.11,
     0.4: 1.19,
     0.6: 1.30,
     0.8: 1.37,
     1.0: 1.48
}

# Initial risk weights — 4 asset classes: 65 / 10 / 5 / 10
eq_risk_weight  = 65   # Equities
bnd_risk_weight = 10   # Bonds
pm_risk_weight  = 5    # Precious Metals
cmd_risk_weight = 10   # Commodities (broad: energy ~30%, agri ~20%, metals)

# Regional risk weights for equity/bonds (unchanged)
regional_risk_weights_data = {
    'category': [
        'Developed_AmericasandUK',
        'Developed_EMEA',
        'Developed_APAC',
        'Emerging_Americas',
        'Emerging_APACandEMEA'
    ],
    'allocation': [10, 35, 35, 10, 10],
}
regional_risk_weights = pd.DataFrame(regional_risk_weights_data)

# Intra-asset Sharpe adjustment tables
# Equities: ±0.1 sensitivity
intra_asset_equity_trailing_sr_data = {
    -0.1: 0.6, -0.08: 0.66, -0.06: 0.77, -0.04: 0.85, -0.02: 0.94,
     0:   1.0,  0.02: 1.11,  0.04: 1.19,  0.06: 1.30,  0.08: 1.37, 0.1: 1.48
}

# Bonds: ±0.25 sensitivity
intra_asset_bond_trailing_sr_data = {
    -0.25: 0.6, -0.2: 0.66, -0.15: 0.77, -0.1: 0.85, -0.05: 0.94,
     0:    1.0,  0.05: 1.11,  0.1: 1.19,  0.15: 1.30,  0.2: 1.37, 0.25: 1.48
}

# Precious Metals: ±0.15 sensitivity
intra_asset_pm_trailing_sr_data = {
    -0.15: 0.6, -0.12: 0.66, -0.09: 0.77, -0.06: 0.85, -0.03: 0.94,
     0:    1.0,  0.03: 1.11,  0.06: 1.19,  0.09: 1.30,  0.12: 1.37, 0.15: 1.48
}

# Commodities: ±0.15 sensitivity
intra_asset_cmd_trailing_sr_data = {
    -0.15: 0.6, -0.12: 0.66, -0.09: 0.77, -0.06: 0.85, -0.03: 0.94,
     0:    1.0,  0.03: 1.11,  0.06: 1.19,  0.09: 1.30,  0.12: 1.37, 0.15: 1.48
}

# Map asset class → intra-asset SR table
sr_data_map = {
    'equity':         intra_asset_equity_trailing_sr_data,
    'bonds':          intra_asset_bond_trailing_sr_data,
    'preciousMetals': intra_asset_pm_trailing_sr_data,
    'commodities':    intra_asset_cmd_trailing_sr_data,
}


## Performance Data Collection

The following code block retrieves and calculates key performance metrics for all four benchmark ETFs:

| Asset Class | Benchmark | Ticker |
|---|---|---|
| Equities | Vanguard FTSE Dev World | VEVE.L |
| Bonds | SPDR Bloomberg 0–3Y US Agg | SAAA.L |
| Precious Metals | iShares Physical Gold ETC | SGLN.L |
| Commodities | Invesco Bloomberg Commodity | CMOP.L |

Metrics collected for 2025: total return, annualised volatility, and Sharpe ratio. These form the baseline for Sharpe ratio adjustment calculations.

In [3]:
# Calculate benchmark returns and volatility for 2025 — all 4 asset classes
benchmark_year_start = "2025-01-01"
benchmark_year_end   = "2025-12-31"

def _benchmark_metrics(symbol, start, end):
    prices = provider.get_historical_prices(symbol, start_date=start, end_date=end)
    yr = prices["close"]
    ret = round((yr.iloc[-1] - yr.iloc[0]) / yr.iloc[0] * 100, 2)
    vol = round(calculate_annualized_volatility(yr) * 100, 2)
    sr  = round(ret / vol, 2)
    return ret, vol, sr

eq_benchmark_return,  eq_benchmark_volatility,  eq_sharpe_ratio  = _benchmark_metrics("VEVE", benchmark_year_start, benchmark_year_end)
bnd_benchmark_return, bnd_benchmark_volatility, bond_sharpe_ratio = _benchmark_metrics("SAAA", benchmark_year_start, benchmark_year_end)
pm_benchmark_return,  pm_benchmark_volatility,  pm_sharpe_ratio  = _benchmark_metrics("SGLN", benchmark_year_start, benchmark_year_end)
cmd_benchmark_return, cmd_benchmark_volatility, cmd_sharpe_ratio = _benchmark_metrics("CMOP", benchmark_year_start, benchmark_year_end)

print(f"2025 VEVE  return: {eq_benchmark_return}%,  vol: {eq_benchmark_volatility}%,  Sharpe: {eq_sharpe_ratio}")
print(f"2025 SAAA  return: {bnd_benchmark_return}%, vol: {bnd_benchmark_volatility}%, Sharpe: {bond_sharpe_ratio}")
print(f"2025 SGLN  return: {pm_benchmark_return}%,  vol: {pm_benchmark_volatility}%,  Sharpe: {pm_sharpe_ratio}")
print(f"2025 CMOP  return: {cmd_benchmark_return}%, vol: {cmd_benchmark_volatility}%, Sharpe: {cmd_sharpe_ratio}")

2025 VEVE  return: 12.76%,  vol: 14.71%,  Sharpe: 0.87
2025 SAAA  return: 2.82%, vol: 4.94%, Sharpe: 0.57
2025 SGLN  return: 48.91%,  vol: 17.65%,  Sharpe: 2.77
2025 CMOP  return: 5.56%, vol: 13.46%, Sharpe: 0.41


## Weight Interpolation Function

This helper function implements the core logic for Sharpe ratio adjustments:

1. Takes a Sharpe ratio and the asset class sensitivity table as input
2. Maps the ratio to the appropriate adjustment factor via linear interpolation
3. Handles edge cases (above/below table range) by clamping to boundary values
4. Returns the interpolated weight adjustment factor

The same function is reused for all five asset classes; only the sensitivity table (±0.1 / ±0.25 / ±0.15) differs.

In [4]:
# interpolate_adjustment_factor is imported from etf_utils.metrics
# Example usage:
# factor = interpolate_adjustment_factor(sharpe_ratio, trailing_sr_adjustment_factors)

## Final Portfolio Construction

The following calculations implement the complete portfolio construction process:

1. **Intra-class Risk Weight Calculation**
   - Compute median Sharpe ratio per asset class
   - Apply `interpolate_adjustment_factor` to each ETF relative to its class median
   - Normalise adjusted weights within each asset class

2. **Cross-class Allocation**
   - Apply strategic class weights: 65 / 10 / 5 / 5 / 5
   - Combine into a single ranked portfolio DataFrame

3. **Cash Weight Determination**
   - Convert risk weights to cash weights adjusted for volatility

4. **Final Allocation**
   - Calculate GBP investment amounts
   - Ensure diversification across all five asset classes
   - Save to `portfolios` table (year = 2026) and `final_portfolio.csv`

In [5]:
# ── Top-level asset-class weight normalisation (4 classes) ──────────────────────
eq_adj_factor  = interpolate_adjustment_factor(eq_sharpe_ratio,  trailing_sr_adjustment_factors)
bond_adj_factor = interpolate_adjustment_factor(bond_sharpe_ratio, trailing_sr_adjustment_factors)
pm_adj_factor  = interpolate_adjustment_factor(pm_sharpe_ratio,  trailing_sr_adjustment_factors)
cmd_adj_factor = interpolate_adjustment_factor(cmd_sharpe_ratio, trailing_sr_adjustment_factors)

eq_adj  = eq_risk_weight  * eq_adj_factor
bnd_adj = bnd_risk_weight * bond_adj_factor
pm_adj  = pm_risk_weight  * pm_adj_factor
cmd_adj = cmd_risk_weight * cmd_adj_factor

total_adj = eq_adj + bnd_adj + pm_adj + cmd_adj
normalized_eq_weight   = round(eq_adj  / total_adj * 100, 2)
normalized_bond_weight = round(bnd_adj / total_adj * 100, 2)
normalized_pm_weight   = round(pm_adj  / total_adj * 100, 2)
normalized_cmd_weight  = round(cmd_adj / total_adj * 100, 2)
total_normalized_weight = normalized_eq_weight + normalized_bond_weight + normalized_pm_weight + normalized_cmd_weight

# Volatility-adjusted cash weights per asset class
eq_cash_weights  = normalized_eq_weight  / eq_benchmark_volatility
bond_cash_weights = normalized_bond_weight / bnd_benchmark_volatility
pm_cash_weights  = normalized_pm_weight  / pm_benchmark_volatility
cmd_cash_weights = normalized_cmd_weight / cmd_benchmark_volatility

total_cash_weight     = eq_cash_weights + bond_cash_weights + pm_cash_weights + cmd_cash_weights
final_eq_cash_weight   = total_normalized_weight * eq_cash_weights  / total_cash_weight
final_bond_cash_weight = total_normalized_weight * bond_cash_weights / total_cash_weight
final_pm_cash_weight   = total_normalized_weight * pm_cash_weights  / total_cash_weight
final_cmd_cash_weight  = total_normalized_weight * cmd_cash_weights / total_cash_weight

print("Asset Class Cash Weights (volatility-adjusted):")
print(f"  Equities:        {final_eq_cash_weight:.2f}%")
print(f"  Bonds:           {final_bond_cash_weight:.2f}%")
print(f"  Precious Metals: {final_pm_cash_weight:.2f}%")
print(f"  Commodities:     {final_cmd_cash_weight:.2f}%")
total_check = final_eq_cash_weight + final_bond_cash_weight + final_pm_cash_weight + final_cmd_cash_weight
print(f"  Total:           {total_check:.2f}%")

Asset Class Cash Weights (volatility-adjusted):
  Equities:        61.45%
  Bonds:           25.64%
  Precious Metals: 4.14%
  Commodities:     8.77%
  Total:           100.00%


In [6]:
etfs = etfs_df.copy()

# ── Region category weight ──────────────────────────────────────────────────────
# Precious Metals and Commodities use a single Global region → 100% weight
_regional_lookup = regional_risk_weights.set_index('category')['allocation']

def _get_region_weight(row):
    if row['region_category'] == 'Global':
        return 100
    return _regional_lookup.get(row['region_category'], 0)

etfs['region_category_weight'] = etfs.apply(_get_region_weight, axis=1)

# ── Intra-region category weight ───────────────────────────────────────────────
etfs['intra_region_category_weight'] = 0

for asset in etfs['asset_class'].unique():
    for region_category in etfs[etfs['asset_class'] == asset]['region_category'].unique():
        mask = (etfs['asset_class'] == asset) & (etfs['region_category'] == region_category)
        num_categories = len(etfs[mask]['intra_region_category'].unique())
        if num_categories > 0:
            etfs.loc[mask, 'intra_region_category_weight'] = 100 // num_categories

# ── Sharpe ratio calculations ──────────────────────────────────────────────────
etfs['starting_risk_weights'] = etfs['region_category_weight'] * etfs['intra_region_category_weight'] / 100
etfs['yield'] = (etfs['last_year_dividends'] - etfs['ter']).round(2)
etfs['sharpe_ratio'] = (etfs['yield'] / etfs['last_year_volatility']).round(2)

# Accumulating ETCs (PM, commodities) have no dividend → last_year_dividends is NaN
# → sharpe_ratio would also be NaN, silently zeroing all downstream risk weights.
# Fall back to last_year_return_per_risk which is already a risk-adjusted return.
etfs['sharpe_ratio'] = etfs['sharpe_ratio'].fillna(etfs['last_year_return_per_risk'])

# Median Sharpe per asset class
median_sharpe = {
    ac: etfs[etfs['asset_class'] == ac]['sharpe_ratio'].median()
    for ac in etfs['asset_class'].unique()
}

etfs['relative_sharpe_ratio'] = etfs.apply(
    lambda row: row['sharpe_ratio'] - median_sharpe[row['asset_class']], axis=1
)
etfs['interpolated_adjusted_factors'] = etfs.apply(
    lambda row: interpolate_adjustment_factor(
        row['relative_sharpe_ratio'], sr_data_map[row['asset_class']]
    ), axis=1
)

# ── Risk weights ───────────────────────────────────────────────────────────────
etfs['adjusted_risk_weights'] = (
    etfs['interpolated_adjusted_factors'] * etfs['starting_risk_weights']
).round(2)

# Normalize within each asset class to sum to 100%
etfs['normalized_risk_weights'] = etfs.groupby('asset_class')['adjusted_risk_weights'].transform(
    lambda x: x / x.sum() * 100
)

# ── Cash weights (intra-asset class) ──────────────────────────────────────────
etfs['cash_weights_guess'] = etfs['normalized_risk_weights'] / etfs['last_year_volatility']

etfs['cash_weights'] = etfs.groupby('asset_class')['cash_weights_guess'].transform(
    lambda x: (etfs.loc[x.index, 'normalized_risk_weights'].sum() / x.sum()) * x
).round(2)

# Final regional category weight (sum of cash weights per region per asset class)
etfs['region_category_weight_final'] = etfs.groupby(
    ['asset_class', 'region_category']
)['cash_weights'].transform('sum')

# ── Asset class weight map (4 classes) ────────────────────────────────────────
asset_class_weight_map = {
    'equity':         normalized_eq_weight,
    'bonds':          normalized_bond_weight,
    'preciousMetals': normalized_pm_weight,
    'commodities':    normalized_cmd_weight,
}
etfs['normalized_asset_class_weight'] = etfs['asset_class'].map(asset_class_weight_map)

# ── Final weights (all 4 asset classes together) ──────────────────────────────
etfs['final_risk_weights'] = etfs['normalized_asset_class_weight'] * etfs['normalized_risk_weights'] / 100
etfs['final_cash_weights_guess'] = etfs['final_risk_weights'] / etfs['last_year_volatility']
etfs['final_cash_weights'] = (
    etfs['final_risk_weights'].sum() / etfs['final_cash_weights_guess'].sum()
    * etfs['final_cash_weights_guess']
).round()

# ── Investment amounts (GBP) ───────────────────────────────────────────────────
etfs['investment'] = 20000 * etfs['final_cash_weights'] / 100

# ── Per-platform weights (each platform sums to 100%, whole numbers) ──────────
# Uses largest-remainder method to guarantee exact 100% sum after rounding.
def _largest_remainder_round(series):
    """Round a Series to whole numbers that sum to exactly 100."""
    raw = series / series.sum() * 100
    floored = np.floor(raw).astype(int)
    remainders = raw - floored
    shortfall = 100 - floored.sum()
    # Award +1 to the largest remainders until we hit 100
    top_idx = remainders.nlargest(int(shortfall)).index
    floored.loc[top_idx] += 1
    return floored

etfs['platform_weight'] = 0
for platform in etfs['platform'].unique():
    mask = etfs['platform'] == platform
    etfs.loc[mask, 'platform_weight'] = _largest_remainder_round(
        etfs.loc[mask, 'final_cash_weights']
    )

# ── Save to DB (year=2026, overwritable) and CSV backup ──────────────────────
save_portfolio(etfs, year=2026)
etfs.to_csv(DATA_OUTPUT / 'final_portfolio.csv', index=False)

# 2025 calendar-year return sparse in equity/bond raw data; fall back to
# trailing 12-month return (last_year) so the display column is always populated.
etfs['2025'] = etfs['2025'].fillna(etfs['last_year'])

etfs.sort_values(by=['asset_class', 'investment'], ascending=[False, False])[[
    'asset_class', 'region_category', 'country', 'intra_region_category',
    'ticker', 'name', 'final_cash_weights', 'platform_weight',
    '2025', 'yday_close_price', 'ter', 'investment', 'platform'
]]